# mmxai-lab — chạy trên Colab (dự phòng khi hết GPU Kaggle)
Runtime → Change runtime type → GPU. Điền ô dưới rồi Runtime → Run all. Kết quả lưu vào Drive `paper-lab/mmxai-lab/runs/`;
về máy: tải `runs.zip` rồi `make import SRC=~/Downloads/runs.zip`.

In [ ]:
REPO_URL = "https://github.com/<owner>/mmxai-lab"   # repo project (private: dán token GitHub khi được hỏi)
BRANCH = "main"
EXP = "smoke"
ARGS = "--limit 4 --tag colab"
import os, getpass, subprocess
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/paper-lab/mmxai-lab/runs'
os.makedirs(RUNS, exist_ok=True)
url = REPO_URL
if 'github.com' in url and not os.path.exists('/content/mmxai-lab'):
    tok = getpass.getpass('GitHub token (Enter nếu repo public): ')
    if tok: url = url.replace('https://', f'https://{tok}@')
    subprocess.run(['git', 'clone', '-q', '--recurse-submodules', '-b', BRANCH, url, '/content/mmxai-lab'], check=True)
%cd /content/mmxai-lab
!git submodule update --init --recursive -q
!pip install -q -e . && (test -f configs/vendor.yaml && python - <<'PY'
import yaml, subprocess, os
v = yaml.safe_load(open('configs/vendor.yaml'))
if v.get('pip'): subprocess.run(['pip', 'install', '-q', *v['pip']], check=True)
for cmd in v.get('install') or []: subprocess.run(cmd, shell=True, cwd=f"vendor/{v['name']}", check=True)
PY
)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import os
os.environ['LAB_USER'] = 'huy'
!python -m mmxai.run --exp $EXP $ARGS --runs-dir $RUNS --no-publish
!cd $RUNS/.. && zip -qr runs.zip runs && ls -la runs.zip